# Phase 5 — Freelancer–Project Matching Model

This notebook mirrors the reproducible training script in `ML/scripts/train_model.py`. It loads the synthetic dataset, trains a Random Forest, evaluates it on a stratified hold-out split, runs 5-fold cross-validation, and saves the model artifacts.

> **Synthetic-data disclaimer:** The dataset is synthetic and its labels are generated by a transparent rubric. The metrics below do not establish real-world matching accuracy.


## 1. Setup


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ML.features import FEATURE_COLUMNS, LABELS
from ML.scripts.train_model import (
    DEFAULT_DATASET,
    DEFAULT_METADATA,
    DEFAULT_METRICS,
    DEFAULT_MODEL,
    DEFAULT_REPORT,
    load_dataset,
    save_artifacts,
    train_and_evaluate,
    write_report,
)

print("Project root:", PROJECT_ROOT)
print("Features:", FEATURE_COLUMNS)
print("Labels:", LABELS)


## 2. Load and validate the synthetic dataset


In [ ]:
dataframe = load_dataset(DEFAULT_DATASET)
print("Rows:", len(dataframe))
print("Columns:", list(dataframe.columns))
print("\nLabel distribution:")
print(dataframe["label"].value_counts().to_string())
dataframe.head()


## 3. Train and evaluate

`train_and_evaluate` validates the dataset, performs a stratified 80/20 split, fits the pipeline (median imputer + Random Forest), computes test metrics, and runs 5-fold stratified cross-validation on the training split.


In [ ]:
pipeline, result = train_and_evaluate(dataframe)
print("Model:", result["model_type"], result["model_version"])
print("Train/test rows:", result["split"]["train_rows"], result["split"]["test_rows"])
print("Accuracy:", round(result["metrics"]["accuracy"] * 100, 2), "%")
print("Macro precision:", round(result["metrics"]["precision_macro"] * 100, 2), "%")
print("Macro recall:", round(result["metrics"]["recall_macro"] * 100, 2), "%")
print("Macro F1:", round(result["metrics"]["f1_macro"] * 100, 2), "%")


## 4. Confusion matrix and per-class report


In [ ]:
import pandas as pd

confusion = pd.DataFrame(
    result["metrics"]["confusion_matrix"],
    index=[f"actual: {label}" for label in LABELS],
    columns=[f"predicted: {label}" for label in LABELS],
)
confusion


In [ ]:
report = pd.DataFrame(result["metrics"]["classification_report"]).transpose()
report


## 5. Cross-validation and feature importance


In [ ]:
pd.DataFrame(result["cross_validation"]).transpose()


In [ ]:
importance = pd.Series(result["feature_importances"]).sort_values(ascending=False)
importance.to_frame("importance")


## 6. Save the model and report

The backend loads `ML/models/match_model.joblib` once through a lock-protected singleton. Metadata records the feature order, classes, metrics and rubric.


In [ ]:
save_artifacts(
    pipeline,
    result,
    model_path=DEFAULT_MODEL,
    metadata_path=DEFAULT_METADATA,
    metrics_path=DEFAULT_METRICS,
)
write_report(result, report_path=DEFAULT_REPORT)
print("Saved model:", DEFAULT_MODEL)
print("Saved metadata:", DEFAULT_METADATA)
print("Saved report:", DEFAULT_REPORT)


## Notes

- Application status, hiring outcome and future information are never model inputs.
- `match_score` is used only to derive the synthetic label; it is not a model input.
- The imputer is inside the pipeline, so it is fitted on training data only.
- Reported metrics are from the actual run, but they are synthetic-data metrics and must not be presented as real-world accuracy.
